In [ ]:
%sql
/*
  SQL script for testing transformation and validation of delivery_dt column
  in the purgo_playground.purgo_playground.f_order table.
  This script includes checks for data type compliance, correct format,
  and integration with external datasets.
*/

-- Setup: Create the target order table schema without constraints due to syntax support issues
CREATE TABLE IF NOT EXISTS purgo_playground.purgo_playground.f_order (
  order_nbr STRING,
  order_type BIGINT,
  delivery_dt DECIMAL(38,0),  -- Delivery date in yyyymmdd format
  order_qty DOUBLE,
  sched_dt DECIMAL(38,0),
  expected_shipped_dt DECIMAL(38,0),
  actual_shipped_dt DECIMAL(38,0),
  order_line_nbr STRING,
  loc_tracker_id STRING,
  shipping_add STRING,
  primary_qty DOUBLE,
  open_qty DOUBLE,
  shipped_qty DOUBLE,
  order_desc STRING,
  flag_return STRING,
  flag_cancel STRING,
  cancel_dt DECIMAL(38,0),
  cancel_qty DOUBLE,
  crt_dt TIMESTAMP,
  updt_dt TIMESTAMP
);

-- Test: Validate delivery_dt format and datatype
WITH Validation AS (
  SELECT delivery_dt
  FROM purgo_playground.purgo_playground.f_order
  WHERE CAST(delivery_dt AS STRING) NOT LIKE '____/__/__'  -- yyyymmdd pattern with "_" as placeholders
)
SELECT COUNT(*) AS invalid_count
FROM Validation
WHERE delivery_dt BETWEEN 20200101 AND 20501231;

-- Integration Test: Verify insertion into supply_chain_delivery table
INSERT INTO purgo_playground.purgo_playground.supply_chain_delivery
SELECT order_nbr AS Shipment_ID, shipping_add AS Warehouse, 'Delivered' AS Delivery_Status
FROM purgo_playground.purgo_playground.f_order
WHERE delivery_dt BETWEEN 20200101 AND 20501231;

-- Test: Confirm successful conversion of delivery_dt to Decimal(38,0)
SELECT delivery_dt
FROM purgo_playground.purgo_playground.f_order
WHERE delivery_dt BETWEEN 20240101 AND 20241231;

-- Test: Perform MERGE operation on f_order table
MERGE INTO purgo_playground.purgo_playground.f_order AS target
USING (
  SELECT order_nbr, delivery_dt
  FROM purgo_playground.purgo_playground.f_order
  WHERE order_qty > 10
) AS source
ON target.order_nbr = source.order_nbr
WHEN MATCHED THEN
  UPDATE SET delivery_dt = source.delivery_dt + 1;

-- Test: Validate DELETE operation
DELETE FROM purgo_playground.purgo_playground.f_order
WHERE shipped_qty < 0;

-- Performance Test: Measure execution time for SELECT query
SELECT delivery_dt
FROM purgo_playground.purgo_playground.f_order
WHERE order_qty > 10;

-- Cleanup: Remove invalid records for reporting consistency
DELETE FROM purgo_playground.purgo_playground.f_order
WHERE delivery_dt IS NULL OR CAST(delivery_dt AS STRING) NOT LIKE '____/__/__';
